# StyleSense — Project Exploration Notebook

Use this notebook to:
- Visualize CNN embeddings with t-SNE
- Test the recommendation pipeline
- Analyze color harmony
- Generate GradCAM heatmaps
- Evaluate outfit scores

In [ ]:
import sys, os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

print('Imports OK')

## 1. CNN Feature Extraction

In [ ]:
from models.cnn_extractor import ClothingCNNExtractor

extractor = ClothingCNNExtractor()

# Test on a sample image
# img_path = '../data/raw_images/tops/sample.jpg'
# preds = extractor.classify(img_path)
# print('Category:', preds['category'])
# print('Formality:', preds['formality'])
# print('Season:', preds['season'])
# print('Embedding shape:', preds['embedding'].shape)

print('CNN extractor ready. Point to an image to test.')

## 2. t-SNE Visualization of Embeddings

In [ ]:
from sklearn.manifold import TSNE
from pathlib import Path
import pandas as pd

catalog_csv = Path('../data/catalog.csv')
emb_dir = Path('../data/embeddings')

if catalog_csv.exists():
    df = pd.read_csv(catalog_csv)
    
    # Load embeddings
    embs, cats = [], []
    for _, row in df.iterrows():
        ep = Path(str(row.get('embedding_path', '')))
        if ep.exists():
            embs.append(np.load(ep))
            cats.append(row['category'])
    
    if len(embs) > 10:
        X = np.stack(embs)
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embs)-1))
        X_2d = tsne.fit_transform(X)
        
        # Plot
        cat_list = sorted(set(cats))
        colors = plt.cm.Set1(np.linspace(0, 1, len(cat_list)))
        color_map = dict(zip(cat_list, colors))
        
        fig, ax = plt.subplots(figsize=(10, 8))
        for i, (x, c) in enumerate(zip(X_2d, cats)):
            ax.scatter(x[0], x[1], color=color_map[c], alpha=0.6, s=20)
        
        patches = [mpatches.Patch(color=color_map[c], label=c) for c in cat_list]
        ax.legend(handles=patches)
        ax.set_title('CNN Embedding Space (t-SNE) — Colored by Category')
        plt.tight_layout()
        plt.show()
    else:
        print('Not enough embeddings yet. Run data/prepare_dataset.py first.')
else:
    print('No catalog.csv found. Run data/prepare_dataset.py first.')

## 3. Color Analysis Demo

In [ ]:
from utils.color_analyzer import DominantColorExtractor, ColorHarmonyChecker

# Simulate outfit with known colors (for testing without images)
# Each item gets a list of dominant color results
mock_outfit_colors = [
    # Item 1: white shirt
    [{'name': 'white', 'family': 'neutral', 'proportion': 0.8,
      'rgb': (255,255,255), 'hex': '#FFFFFF', 'hsl': (0,0,100)}],
    # Item 2: navy trousers  
    [{'name': 'navy', 'family': 'cool', 'proportion': 0.9,
      'rgb': (20,30,100), 'hex': '#141E64', 'hsl': (234,67,24)}],
    # Item 3: brown loafers
    [{'name': 'brown', 'family': 'earth', 'proportion': 0.7,
      'rgb': (120,70,30), 'hex': '#78461E', 'hsl': (27,60,29)}],
]

checker = ColorHarmonyChecker()
result = checker.check(mock_outfit_colors)

print(f'Color Score: {result["score"]}/100')
print(f'Pass: {result["pass"]}')
print(f'Colors: {result["primary_colors"]}')
print(f'Families: {result["families"]}')
print(f'Positives: {result["positives"]}')
print(f'Issues: {result["issues"]}')
print(f'Explanation: {result["explanation"]}')

# Visualize palette
fig, axes = plt.subplots(1, 3, figsize=(6, 2))
for ax, item_colors in zip(axes, mock_outfit_colors):
    rgb = [c['rgb'] for c in item_colors]
    for i, (r, g, b) in enumerate(rgb[:1]):
        ax.add_patch(plt.Rectangle((0,0),1,1, color=(r/255,g/255,b/255)))
    ax.set_title(item_colors[0]['name'], fontsize=9)
    ax.axis('off')
plt.suptitle(f'Outfit Palette — Score: {result["score"]}/100', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Full Evaluation Pipeline Demo

In [ ]:
from utils.outfit_evaluator import OutfitEvaluator, ClothingItem

evaluator = OutfitEvaluator()

# Test outfit: smart casual for college
outfit = [
    ClothingItem(name='White Linen Shirt', category='top',    color_name='white', fabric='linen',  formality=0.55),
    ClothingItem(name='Slim Navy Chinos',  category='bottom', color_name='navy',  fabric='cotton', formality=0.55),
    ClothingItem(name='White Sneakers',    category='shoes',  color_name='white', fabric='canvas', formality=0.30),
    ClothingItem(name='Minimalist Watch',  category='accessory', color_name='silver', fabric='metal', formality=0.60),
]

result = evaluator.evaluate(
    items=outfit,
    occasion='college',
    style_pref='minimalist',
    season='summer',
)

print(f'\n{'='*50}')
print(f'  OUTFIT EVALUATION')
print(f'{'='*50}')
print(f'  Overall Score: {result.overall_score}/100')
print(f'  Verdict: {result.verdict}')
print(f'  Explanation: {result.explanation}')
print(f'\n  Check Scores:')
for check_name, check_data in result.checks.items():
    icon = '✅' if check_data['pass'] else '❌'
    print(f'  {icon} {check_name.replace("_"," ").title()}: {check_data["score"]}/100')
print(f'\n  Suggestions:')
for s in result.suggestions:
    print(f'  • {s}')
print(f'\n  Alternatives:')
for a in result.alternative_items:
    print(f'  → Replace "{a["replace"]}" with "{a["with"]}"')
    print(f'    Reason: {a["reason"]}')

## 5. Recommendation System Demo

In [ ]:
from utils.recommender import OutfitRecommender

recommender = OutfitRecommender()

outfits = recommender.recommend(
    occasion='college',
    style='minimalist',
    season='summer',
    n_outfits=3,
)

for i, outfit in enumerate(outfits, 1):
    print(f'\nOutfit {i}:')
    for item in outfit:
        print(f'  [{item.category.upper():12s}] {item.name} ({item.color_name}, {item.fabric}, formality={item.formality:.1f})')